# What Is a Neighbor? Spatial Weights

**Series:** Spatial Autocorrelation with PySAL (notebook 1 of 3)
**Time:** about 25 minutes

**In this notebook you will:**
- Load Indiana county data from the Census
- Learn three ways to define neighbors: Rook, Queen and K-nearest
- Build a spatial weights object with PySAL and look inside it
- Draw the neighbor connections on a map

**You should already know:** GeoDataFrames and joins (Foundations notebooks 4a to 4c) and Census API basics (Data Collection notebooks 1a to 1c).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Why neighbors matter

**Tobler's first law of geography:** "everything is related to everything else, but near things
are more related than distant things."

Before we can test whether nearby counties are similar, we must tell the computer which counties
count as **neighbors**. That list of neighbors is called a **spatial weights** object, often
written **W**.

## Step 1: Install and import

In [ ]:
!pip install -q libpysal esda splot mapclassify

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

## Step 2: Get the data

We use the **American Community Survey (ACS) 5-year estimates, 2023** for Indiana's 92 counties:

| Census variable | Meaning | Our column |
|---|---|---|
| `B01003_001E` | Total population | `total_pop` |
| `B01002_001E` | Median age | `median_age` |
| `B02001_005E` | Asian alone population | `asian_pop` |

> **Every learner needs their own free Census API key.** It is the same key used in the Census data
> session. If you do not have one yet, request it at <https://api.census.gov/data/key_signup.html>;
> it arrives by email within a few minutes and must be activated from the link in that email.

Paste your key between the quotes below. The data and the county boundaries are read straight from
the Census website into memory, so there is nothing to download.

In [ ]:
CENSUS_KEY = "PASTE_YOUR_CENSUS_API_KEY_HERE"

def load_indiana_counties(key):
    if key.startswith("PASTE"):
        raise ValueError("Paste your free Census API key into CENSUS_KEY above, then run this cell again.")
    # 1. Attributes from the Census API
    reply = requests.get(
        "https://api.census.gov/data/2023/acs/acs5",
        params={
            "get": "NAME,B01003_001E,B01002_001E,B02001_005E",
            "for": "county:*",
            "in": "state:18",          # 18 = Indiana
            "key": key,
        },
        timeout=60,
    )
    reply.raise_for_status()
    rows = reply.json()
    table = pd.DataFrame(rows[1:], columns=rows[0])
    table["GEOID"] = table["state"] + table["county"]
    table = table.rename(columns={"B01003_001E": "total_pop",
                                  "B01002_001E": "median_age",
                                  "B02001_005E": "asian_pop"})
    for col in ["total_pop", "median_age", "asian_pop"]:
        table[col] = pd.to_numeric(table[col])
    table["pct_asian"] = 100 * table["asian_pop"] / table["total_pop"]

    # 2. County boundaries from the Census cartographic boundary files
    shapes = gpd.read_file(
        "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip")
    shapes = shapes[shapes["STATEFP"] == "18"][["GEOID", "geometry"]]

    # 3. Join, then project to meters (UTM zone 16N) for spatial analysis
    counties = shapes.merge(table, on="GEOID").to_crs(epsg=26916)
    return counties.reset_index(drop=True)

counties = load_indiana_counties(CENSUS_KEY)
print(len(counties), "counties loaded")
counties[["NAME", "total_pop", "median_age", "pct_asian"]].head()

## Step 3: Look at the map

In [ ]:
counties.plot(edgecolor="black", facecolor="lightgray", figsize=(6, 8))
plt.title("Indiana counties")
plt.axis("off")
plt.show()

## Step 4: Three ways to define neighbors

Think of a chess board:

| Rule | Who is a neighbor? | Chess piece |
|---|---|---|
| **Rook** | Shares an **edge** (a border line) | moves straight |
| **Queen** | Shares an edge **or a corner** | moves straight or diagonal |
| **K-nearest (KNN)** | The *k* closest counties by center point | ignores borders |

Queen always finds at least as many neighbors as Rook.

In [ ]:
from libpysal.weights import Rook, Queen, KNN

w_rook = Rook.from_dataframe(counties, use_index=True)
w_queen = Queen.from_dataframe(counties, use_index=True)
w_knn = KNN.from_dataframe(counties, k=4)

print("Average number of neighbors")
print("  Rook :", round(w_rook.mean_neighbors, 2))
print("  Queen:", round(w_queen.mean_neighbors, 2))
print("  KNN  :", round(w_knn.mean_neighbors, 2))

## Step 5: Look inside W

`w.neighbors` is a dictionary: each county's row number points to a list of its neighbors' row numbers.

In [ ]:
i = counties.index[counties["NAME"].str.startswith("Tippecanoe")][0]
print("Tippecanoe County is row", i)
print("Rook neighbors :", sorted(w_rook.neighbors[i]))
print("Queen neighbors:", sorted(w_queen.neighbors[i]))
print("Names:", list(counties.loc[w_queen.neighbors[i], "NAME"].str.replace(", Indiana", "")))

## Step 6: Draw the connections

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8))
for ax, w, title in [(axes[0], w_rook, "Rook"), (axes[1], w_queen, "Queen")]:
    counties.plot(ax=ax, edgecolor="gray", facecolor="white")
    w.plot(counties, ax=ax, edge_kws={"color": "red", "linewidth": 0.8},
           node_kws={"s": 5, "color": "black"})
    ax.set_title(f"{title}: {w.s0:.0f} links")
    ax.axis("off")
plt.show()

Each red line joins two neighbors. The Queen map has a few extra diagonal links where four
counties meet at a single corner.

## Step 7: Row standardize

Before most statistics we **row standardize** W, so each county's neighbor weights add up to 1.
That way a county with 8 neighbors does not count more than one with 3.

In [ ]:
w_queen.transform = "R"
print("Tippecanoe neighbor weights:", w_queen.weights[i])

## Try it yourself

Build a KNN weights object with `k=6`. Which counties are Tippecanoe's 6 nearest neighbors? Are they the same as its Queen neighbors?

<details>
<summary><b>Show a solution</b></summary>

```python
w_knn6 = KNN.from_dataframe(counties, k=6)
print(sorted(counties.loc[w_knn6.neighbors[i], "NAME"].str.replace(", Indiana", "")))
```

</details>

In [ ]:
# Write your answer here

## Recap

- Spatial weights (W) record which places are neighbors.
- Rook uses shared edges, Queen adds shared corners, KNN uses the k closest centers.
- Row standardizing makes each place's neighbor weights sum to 1.

**Next:** [Global Moran's I](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/02b_Global_Morans_I.ipynb)